In [1]:
# Install LIbraries
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 805.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [2]:
# Imports
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
DPO patch applied.
GPU: Tesla T4


In [8]:
# 5. Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"


def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [9]:
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages


def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)


def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

In [5]:
# 3. File path
non_instruction_data_path = "/content/non_instruction_data.pdf"
instruction_data_path = "/content/instruction_dataset.jsonl"
preference_data_path = "/content/preference_dataset.jsonl"

# for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
for path in [non_instruction_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# 4. Simple config

BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH = 512
SEED = 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 5
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 30
STAGE2_MAX_STEPS = 30
STAGE3_MAX_STEPS = 30

STAGE1_LR = 2e-4
STAGE2_LR = 1e-4
STAGE3_LR = 5e-5

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_merge_reload_outputs"

STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_instruction_adapter"
STAGE2_MERGED_DIR  = f"{OUTPUT_ROOT}/stage2_instruction_merged_model"

STAGE3_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage3_dpo_adapter"
FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage3_dpo_final_merged_model"

for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
    STAGE2_ADAPTER_DIR,
    STAGE2_MERGED_DIR,
    STAGE3_ADAPTER_DIR,
    FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [6]:
# ============================================================
# STAGE 1: Non-instruction continued pretraining
# ============================================================

print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

# Define the stage1_dataset by calling build_pdf_dataset
stage1_dataset = build_pdf_dataset(non_instruction_data_path)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,

    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

del stage1_trainer
del stage1_model
clear_gpu_memory()


STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:21<00:00, 21.48s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_merge_reload_outputs/stage1_non_instruction_merged_model


## Base Model Testing: IT Helpdesk Questions

This section tests the capabilities of the base model (`unsloth/tinyllama-bnb-4bit`) before any instruction-based fine-tuning. We will use a set of IT helpdesk-related questions and evaluate the model's responses.

In [7]:
print("\n==========================")
print("BASE MODEL TESTING")
print("==========================")

# Define a list of 10 IT helpdesk questions
it_helpdesk_questions = [
    "My laptop is not turning on. What should I do?",
    "How do I reset my password for the company's internal network?",
    "I can't access my email. What are the common troubleshooting steps?",
    "My printer is not working. How can I fix it?",
    "How do I connect to the office Wi-Fi?",
    "My computer is running very slowly. What could be the cause?",
    "How do I install new software on my work computer?",
    "I accidentally deleted an important file. Can I recover it?",
    "What should I do if I suspect a phishing email?",
    "How do I request new hardware like a monitor or keyboard?"
]

# Load the base model and tokenizer for inference
# We load the base model directly without LoRA for this test
base_test_model, base_test_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if base_test_tokenizer.pad_token is None:
    base_test_tokenizer.pad_token = base_test_tokenizer.eos_token

base_test_tokenizer.padding_side = "right"

# Prepare to store results
results = []

print(f"\nGenerating answers for {len(it_helpdesk_questions)} questions with the base model...")

# Generate answers for each question
for i, question in enumerate(it_helpdesk_questions):
    print(f"\n--- Question {i+1}/{len(it_helpdesk_questions)} ---")
    print(f"Question: {question}")

    # Use the existing generate_answer function
    answer = generate_answer(
        model=base_test_model,
        tokenizer=base_test_tokenizer,
        instruction=question,
        max_new_tokens=256 # Increased tokens for potentially longer answers
    )

    print(f"Base Model Answer: {answer}")

    results.append({
        "Questions": question,
        "Base Model Answer": answer,
        "Problem": "The answer is generic and not specific to our IT Helpdesk policy"
    })

# Clear GPU memory after inference
del base_test_model
del base_test_tokenizer
clear_gpu_memory()


BASE MODEL TESTING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Generating answers for 10 questions with the base model...

--- Question 1/10 ---
Question: My laptop is not turning on. What should I do?
Base Model Answer: This is a known issue with the current version of the firmware on your device and will be fixed in an upcoming firmware update.

#### How to fix:

If you are using a computer, download the latest firmware from the following link and apply it. 
* http://www.google.com/intl/en_us/chrome/devices/firmware-update/
* https://support.google.com/chromebooks/answer/1052829?hl=

In [10]:
import pandas as pd

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

print("\nBase Model Test Results:")
display(results_df)

print("\nPlease review the 'Base Model Answer' column and fill in any 'Problem' descriptions directly in the DataFrame by double-clicking the cell.")


Base Model Test Results:


,Questions,Base Model Answer,Problem
0,My laptop is not turning on. What should I do?,This is a known issue with the current version...,The answer is generic and not specific to our ...
1,How do I reset my password for the company's i...,* **Service:** Network\n* **Service Type:** Ne...,The answer is generic and not specific to our ...
2,I can't access my email. What are the common t...,"I have no email access. I'm able to log in, bu...",The answer is generic and not specific to our ...
3,My printer is not working. How can I fix it?,This is an error that can be fixed by a restar...,The answer is generic and not specific to our ...
4,How do I connect to the office Wi-Fi?,```\nWelcome to the office Wi-Fi!\n```,The answer is generic and not specific to our ...
5,My computer is running very slowly. What could...,"Your computer is running very slow, but we can...",The answer is generic and not specific to our ...
6,How do I install new software on my work compu...,"1. To install new software, select the appropr...",The answer is generic and not specific to our ...
7,I accidentally deleted an important file. Can ...,"1. No, you can't.\n2. You will be asked to cho...",The answer is generic and not specific to our ...
8,What should I do if I suspect a phishing email?,1. If you are the target of the phishing atte...,The answer is generic and not specific to our ...
9,How do I request new hardware like a monitor o...,If you are having a problem with your hardware...,The answer is generic and not specific to our ...



Please review the 'Base Model Answer' column and fill in any 'Problem' descriptions directly in the DataFrame by double-clicking the cell.


In [11]:
# ============================================================
# STAGE 2 DATA: Instruction JSONL
# ============================================================
'''
Print Statements:
The code starts with print statements (print("\n=============================="), etc.) to clearly indicate that the processing for 'STAGE 2: INSTRUCTION DATA' is beginning.
'''
print("\n==============================")
print("STAGE 2: INSTRUCTION DATA")
print("==============================")

'''
Load Instruction Dataset:
instruction_dataset = load_dataset("json", data_files=instruction_data_path, split="train") loads the instruction dataset. It expects the data to be in JSON format, specified by data_files=instruction_data_path (which points to a JSONL file), and loads the 'train' split of this dataset. This dataset typically contains prompts, inputs, and desired responses for instruction-following.
'''
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train",
)

'''
Validate Required Columns:
The code then defines required_instruction_cols = {"instruction", "output"} and checks if these essential columns (instruction and output) are present in the loaded instruction_dataset. If any are missing, it raises a ValueError, ensuring data integrity before proceeding
'''
required_instruction_cols = {"instruction", "response"} # Changed from 'output' to 'response'
missing_cols = required_instruction_cols - set(instruction_dataset.column_names)

if missing_cols:
    # Diagnostic helper for the user
    print(f"\nERROR: The instruction dataset is missing the following required columns: {missing_cols}")
    print(f"Please ensure your '{instruction_data_path}' file contains these columns.")
    print("Here are the columns found in your dataset:", instruction_dataset.column_names)
    print("--- Sample of first few records from instruction_dataset.jsonl ---")
    try:
        with open(instruction_data_path, 'r') as f:
            for i, line in enumerate(f):
                if i >= 3: # Print first 3 lines
                    break
                print(f"Line {i+1}: {line.strip()}")
    except Exception as e:
        print(f"Could not read sample from {instruction_data_path}: {e}")
    print("------------------------------------------------------------------")
    raise ValueError(f"Instruction dataset missing columns: {missing_cols}")

'''
format_instruction_record(example) Function:
This function is a crucial pre-processing step. It takes a single record (an example from the dataset) and formats it into a single string under the key 'text'. It retrieves the instruction, input_text (if available), and output fields from the example. It then uses the previously defined build_instruction_prompt helper function to create the prompt part (### Instruction: ... ### Input: ... ### Response: ) and appends the output to it. This creates a complete instruction-response pair that the model will learn from.
'''
def format_instruction_record(example):
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("response", "") # Changed from 'output' to 'response'

    text = build_instruction_prompt(instruction, input_text) + str(output).strip()
    return {"text": text}

'''
Apply Formatting to Dataset:
stage2_dataset = instruction_dataset.map(format_instruction_record) applies the format_instruction_record function to every record in the instruction_dataset. This transforms the raw instruction data into the stage2_dataset, where each entry has a text field containing the properly formatted instruction-response string ready for SFTTrainer.
'''
stage2_dataset = instruction_dataset.map(format_instruction_record)

'''
Print Statistics: Finally, it prints the total number of instruction rows (len(stage2_dataset)) and a sample of the formatted instruction text, allowing for a quick verification of the processed data.
'''
print("Instruction rows:", len(stage2_dataset))
print("\nSample instruction text:\n", stage2_dataset[0]["text"][:900])


STAGE 2: INSTRUCTION DATA


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Instruction rows: 101

Sample instruction text:
 ### Instruction:
What is the primary role of the IT Helpdesk?

### Response:
The IT Helpdesk serves as the single point of contact (SPOC) for all end-user technical inquiries, incidents, and service requests within the enterprise environment. [cite: 2]


In [12]:
# ============================================================
# STAGE 2: Load Stage 1 merged model -> instruction SFT
# ============================================================
'''
This code block handles Stage 2: Instruction Fine-Tuning. In this stage, the model, which has already undergone non-instruction pretraining in Stage 1, is further fine-tuned to follow specific instructions. Here's a breakdown:

Print Statements: Similar to previous stages, these provide clear console output indicating the start of 'STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN'.
'''
print("\n===============================================")
print("STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN")
print("===============================================")

'''
Load Stage 1 Merged Model: stage2_model, tokenizer = load_unsloth_model_with_lora(STAGE1_MERGED_DIR) loads the model that was merged after Stage 1. This is crucial because Stage 2 builds upon the knowledge acquired in Stage 1. The load_unsloth_model_with_lora function loads it in 4-bit and attaches a new LoRA adapter, allowing for efficient fine-tuning.
'''
stage2_model, tokenizer = load_unsloth_model_with_lora(STAGE1_MERGED_DIR)

'''
Prepare Model for Training: FastLanguageModel.for_training(stage2_model) sets the model to training mode, and tokenizer.padding_side = "right" configures the tokenizer. Right padding is common for causal language models during training.
'''
FastLanguageModel.for_training(stage2_model)
tokenizer.padding_side = "right"

'''
SFTConfig Configuration: SFTConfig defines the training parameters for Supervised Fine-Tuning (SFT).

output_dir: Specifies where training logs will be saved.
max_steps: Sets the total number of training steps (e.g., STAGE2_MAX_STEPS = 30).
per_device_train_batch_size, gradient_accumulation_steps: Control the effective batch size during training.
learning_rate, warmup_steps: Define the learning schedule.
logging_steps, save_strategy, report_to: Control logging and saving behavior.
fp16, bf16, optim: Configure precision and optimizer type.
dataset_text_field: Specifies the column in stage2_dataset that contains the formatted instruction text.
max_length: Sets the maximum sequence length for the input.
packing=False: This is a key difference from Stage 1. For instruction tuning, we want to preserve the distinct instruction-response pairs and not pack multiple examples into a single sequence, as this could disrupt the instruction-following structure.
seed: For reproducibility.
'''
stage2_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage2_logs",

    max_steps=STAGE2_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE2_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    seed=SEED,
)

'''
Initialize SFTTrainer: stage2_trainer = SFTTrainer(...) initializes the SFTTrainer with the loaded model, tokenizer, the stage2_dataset (which contains the formatted instruction data), and the defined stage2_config.
'''
stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)
'''
Train and Measure: train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION FINE-TUNING") executes the training loop. This helper function also records and prints the training time and GPU memory usage.
'''
train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION FINE-TUNING")

'''
Test Model Answer: print(generate_answer(stage2_model, tokenizer, "Explain about IT Help desk?", max_new_tokens=120)) tests the fine-tuned model's ability to follow instructions by generating a response to a sample prompt. This provides an immediate qualitative assessment of the instruction-tuning effectiveness.
'''
print("\nStage 2 test answer:")
print(generate_answer(stage2_model, tokenizer, "Explain about IT Help desk?", max_new_tokens=120))

'''
Save Adapter and Merged Model: save_adapter_and_merge(...) saves two versions of the trained model:

The LoRA adapter separately to STAGE2_ADAPTER_DIR.
A full merged model (LoRA adapter integrated into the base model weights) to STAGE2_MERGED_DIR. This merged model will serve as the starting point for Stage 3 (DPO).
'''
save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE2_ADAPTER_DIR,
    merged_dir=STAGE2_MERGED_DIR,
    stage_name="Stage 2",
)

'''
Memory Management: del stage2_trainer, del stage2_model, and clear_gpu_memory() are called to free up GPU memory and Python objects, which is important for managing resources, especially in environments with limited memory like Colab, before proceeding to the next stage
'''
del stage2_trainer
del stage2_model
clear_gpu_memory()


STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/101 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 101 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,3.452900
2,3.395000
3,3.677100
4,3.830300
5,3.581900
6,3.500400
7,3.745500
8,3.423300
9,3.542800
10,3.565000



STAGE 2 - INSTRUCTION FINE-TUNING RESULTS
Train time/sec: 83.45
Peak allocated VRAM/GB: 1.005
Peak reserved VRAM/GB: 1.127

Stage 2 test answer:
Helpdesk is a system in which the help desk representatives are responsible for answering the calls and queries of the customers.

### References:

* [1] IT Help Desk | IT Help Desk Meaning & Explanation with Example
* [2] IT Help Desk Meaning & Explanation with Example - Wikipedia

Saving Stage 2 adapter...
Stage 2 adapter saved to: /content/unsloth_merge_reload_outputs/stage2_instruction_adapter

Merging Stage 2 adapter with base model...
Detected local model directory: /content/unsloth_merge_reload_outputs/stage1_non_instruction_merged_model
Copied tokenizer.model from local model directory
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:17<00:00, 17.24s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:49<00:00, 49.33s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_merge_reload_outputs/stage2_instruction_merged_model`
Stage 2 merged model saved to: /content/unsloth_merge_reload_outputs/stage2_instruction_merged_model


## Instruction-Tuned Model Test Results (Fine-tuned Model Only)

This section exclusively evaluates the instruction-tuned model (`STAGE2_MERGED_DIR`) using the specified IT helpdesk questions and presents the results in a table with only the questions and the fine-tuned model's answers.

In [13]:
print("\n==========================")
print("INSTRUCTION-TUNED MODEL TESTING (FINE-TUNED ONLY)")
print("==========================")

# Define the list of IT helpdesk questions as provided by the user
it_helpdesk_questions = [
    "My laptop is not turning on. What should I do?",
    "How do I reset my password for the company's internal network?",
    "I can't access my email. What are the common troubleshooting steps?",
    "My printer is not working. How can I fix it?",
    "How do I connect to the office Wi-Fi?",
    "My computer is running very slowly. What could be the cause?",
    "How do I install new software on my work computer?",
    "I accidentally deleted an important file. Can I recover it?",
    "What should I do if I suspect a phishing email?",
    "How do I request new hardware like a monitor or keyboard?"
]

# Load the instruction-tuned model and tokenizer for inference
instruction_model, instruction_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE2_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if instruction_tokenizer.pad_token is None:
    instruction_tokenizer.pad_token = instruction_tokenizer.eos_token

instruction_tokenizer.padding_side = "right"

finetuned_results = []

print(f"\nGenerating answers for {len(it_helpdesk_questions)} questions with the instruction-tuned model...")

# Generate answers for each question using the instruction-tuned model
for i, question in enumerate(it_helpdesk_questions):
    print(f"\n--- Question {i+1}/{len(it_helpdesk_questions)} ---")
    print(f"Question: {question}")

    answer = generate_answer(
        model=instruction_model,
        tokenizer=instruction_tokenizer,
        instruction=question,
        max_new_tokens=256
    )

    print(f"Fine-tuned Model Answer: {answer}")

    finetuned_results.append({
        "Questions": question,
        "Fine-tuned Model Answer": answer
    })

# Clear GPU memory after inference
del instruction_model
del instruction_tokenizer
clear_gpu_memory()


INSTRUCTION-TUNED MODEL TESTING (FINE-TUNED ONLY)
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Generating answers for 10 questions with the instruction-tuned model...

--- Question 1/10 ---
Question: My laptop is not turning on. What should I do?
Fine-tuned Model Answer: If you have a laptop with Windows 10, press and hold the power button for at least 15 seconds. Then, press the power button again to turn your computer back on. If you continue to experience issues after this procedure, contact our customer service team at https://www.mouser.com/contact/customer-service/. You may also 

In [14]:
import pandas as pd

# Create a DataFrame from the results with only the requested columns
finetuned_results_df = pd.DataFrame(finetuned_results)

print("\nFine-tuned Model Test Results:")
display(finetuned_results_df)



Fine-tuned Model Test Results:


,Questions,Fine-tuned Model Answer
0,My laptop is not turning on. What should I do?,"If you have a laptop with Windows 10, press an..."
1,How do I reset my password for the company's i...,```bash\nPassword reset\n```\n\n### Reset pass...
2,I can't access my email. What are the common t...,*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*...
3,My printer is not working. How can I fix it?,You are not supposed to use printers.
4,How do I connect to the office Wi-Fi?,The office Wi-Fi is available 24/7 in the offi...
5,My computer is running very slowly. What could...,"Computer is slow because of excessive use, the..."
6,How do I install new software on my work compu...,```shell\nsudo apt-get update && sudo apt-get ...
7,I accidentally deleted an important file. Can ...,I will check your file and give you a proper s...
8,What should I do if I suspect a phishing email?,"If you receive an email that seems suspicious,..."
9,How do I request new hardware like a monitor o...,- New hardware can be requested by emailing th...
